# Cross-validation and hyperparameter search

[Validation and model selection](validation_and_model_selection.ipynb) set aside a validation set to compare models. It works, at two costs: the validation rows are rows the model doesn't learn from, and the validation error is measured on a single sample of rows, so it moves when the sample changes. **Cross-validation** reduces both costs by taking turns: every row is used for validation once, and for training the rest of the time. With it, the settings of a model can be searched systematically.

## Taking turns

In ***k*-fold cross-validation**, the training rows are divided into *k* parts of equal size, the *folds*. The model is trained *k* times, each time on all the folds but one, and measured on the one left out. With 5 folds:

| Round | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| --- | --- | --- | --- | --- | --- |
| 1 | **validation** | training | training | training | training |
| 2 | training | **validation** | training | training | training |
| 3 | training | training | **validation** | training | training |
| 4 | training | training | training | **validation** | training |
| 5 | training | training | training | training | **validation** |

The result is *k* validation errors instead of one: their mean is a steadier estimate, and their spread shows how much it would move with a different sample. The cost is *k* trainings instead of one. The test set stays out of all of it.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
len(X_rest), len(X_test)

(16512, 4128)

No validation set is split off this time: cross-validation makes its own folds from the 16,512 rows that aren't the test set. On the gradient boosting model of [Gradient boosting](../algorithms/tree-based/gradient_boosting.ipynb):

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_rest, y_rest, cv=folds,
                         scoring="neg_mean_absolute_error")
scores

array([-33093.633457  , -31298.82743634, -31377.57478989, -31397.99704353,
       -31083.76955568])

The scores are negative because scikit-learn's searches always *maximise* a score, so errors are given as utilities, with their sign flipped: the larger (closer to 0), the better. Turning them back into errors:

In [ ]:
mae_by_fold = -scores
mae_by_fold.round(0), mae_by_fold.mean().round(0), mae_by_fold.std().round(0)

(array([33094., 31299., 31378., 31398., 31084.]),
 np.float64(31650.0),
 np.float64(730.0))

The five folds give errors between about \$31,100 and \$33,100, with a mean of \$31,700 and a spread (standard deviation) of about \$730. A single validation set would have given one of these numbers, and which one would have depended on the split. The mean is steadier, and the spread is a yardstick: a difference between two models smaller than about \$1,000 could be chance.

## Folds that match the problem

`shuffle=True` above was not a detail. Without it, `KFold` cuts the rows in the order they are stored, and these rows are stored county by county: the first ones in Alameda, in the Bay Area, the last ones in Yuba, in the north.

In [ ]:
ordered_folds = KFold(n_splits=5)
-cross_val_score(HistGradientBoostingRegressor(random_state=42), X_rest.sort_index(), y_rest.sort_index(), cv=ordered_folds,
                 scoring="neg_mean_absolute_error").round(0)

array([45263., 40782., 44164., 44656., 46561.])

Without shuffling, the error jumps to between \$40,800 and \$46,600. Each fold is now a block of whole counties, and each round asks the model to predict counties it has never seen, without a single district of those counties among its training rows. That is a much harder question than predicting a district whose neighbours were in the training data, and it is not a wrong one: if the model were going to be used in new counties, these would be the right folds, and the shuffled ones would be optimistic. The folds don't just measure the model; they define what is being measured.

The folds should copy the split that matches production, as [the test set stands for production](overfitting_and_the_split.ipynb#the-test-set-stands-for-production) explained, and scikit-learn has a splitter for each case:

| What production will bring | Splitter | What it does |
| --- | --- | --- |
| new rows like the old ones | `KFold(shuffle=True)` | random folds |
| the same, for classification | `StratifiedKFold(shuffle=True)` | random folds with the same share of each class |
| the future | `TimeSeriesSplit` | each round trains on the rows before a point in time and validates on the rows just after it |
| new entities: patients, houses, places | `GroupKFold` | all the rows of an entity in the same fold |

An integer such as `cv=5` means `KFold`, or `StratifiedKFold` for a classifier, *without* shuffling: pass a splitter whenever the order of the rows means something.

`TimeSeriesSplit`, on twelve rows ordered in time, shows its rounds: the training rows always come before the validation rows.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

pd.DataFrame([
    {"training rows": f"{training[0]}–{training[-1]}", "validation rows": f"{validation[0]}–{validation[-1]}"}
    for training, validation in TimeSeriesSplit(n_splits=4).split(np.arange(12))
], index=pd.RangeIndex(1, 5, name="round"))

,training rows,validation rows
round,,
1,0–3,4–5
2,0–5,6–7
3,0–7,8–9
4,0–9,10–11


## Hyperparameters

A model's **parameters** are learned by `fit`: the weights of a linear model, the splits of a tree. Its **hyperparameters** are chosen before `fit`: the learning rate, the size of the trees, the number of neighbours, the penalty. [Controlling complexity](regularization.ipynb) showed that each model has some, and that the best values depend on the data. Trying values by hand and looking at the errors is slow and easy to bias; a **search** tries them systematically, each one measured by cross-validation.

## Grid search

`GridSearchCV` tries every combination of the values listed. Three learning rates and three tree sizes are nine combinations, each measured on 5 folds: 45 trainings. `n_jobs=-1` runs them in parallel on every core:

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    HistGradientBoostingRegressor(max_iter=300, random_state=42),
    param_grid={"learning_rate": [0.03, 0.1, 0.3], "max_leaf_nodes": [15, 31, 63]},
    cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1,
).fit(X_rest, y_rest)
grid_search.best_params_, -grid_search.best_score_

({'learning_rate': 0.1, 'max_leaf_nodes': 63}, np.float64(30456.196361613507))

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results.assign(mean_MAE=-results["mean_test_score"]).pivot(
    index="param_learning_rate", columns="param_max_leaf_nodes", values="mean_MAE").round(0)

param_max_leaf_nodes,15,31,63
param_learning_rate,,,
0.03,33573.0,31622.0,30638.0
0.10,31554.0,30735.0,30456.0
0.30,32200.0,32147.0,31799.0


The best combination is a learning rate of 0.1 with trees of up to 63 leaves, at about \$30,460, and the largest trees win at every learning rate. 63 is the largest size in the grid, so a second search should try larger trees too, as explained below. Leaving out the slowest one (a learning rate of 0.03 with 15 leaves), the combinations lie within about \$1,750 of each other, between two and three times the spread across folds measured above.

`n_jobs` deserves care: the search runs trainings in parallel, and some models also parallelise inside each training. Both at once multiply the threads competing for the same cores. The usual choice is to parallelise the search and leave the model to one core.

## Randomised search

A grid grows fast: four hyperparameters with five values each are 625 combinations, 3,125 trainings with 5 folds. `RandomizedSearchCV` tries a fixed number of random combinations instead, drawn from distributions. Its budget doesn't grow with the number of hyperparameters, and since usually only a few of them matter much, random combinations try more different values of those few than a grid of the same size would. Continuous settings, such as a learning rate, are drawn from a continuous range instead of a few chosen points; `loguniform` spreads the draws evenly across orders of magnitude, so that 0.01 to 0.03 gets as many tries as 0.1 to 0.3:

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    HistGradientBoostingRegressor(max_iter=300, random_state=42),
    param_distributions={
        "learning_rate": loguniform(0.01, 0.3),
        "max_leaf_nodes": randint(8, 128),
        "min_samples_leaf": randint(5, 100),
        "l2_regularization": loguniform(1e-3, 10),
    },
    n_iter=20, cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1, random_state=42,
).fit(X_rest, y_rest)
random_search.best_params_, -random_search.best_score_

({'l2_regularization': np.float64(0.020237789540485076),
  'learning_rate': np.float64(0.06960091021987622),
  'max_leaf_nodes': 69,
  'min_samples_leaf': 44},
 np.float64(30100.92593726995))

Twenty random combinations of four settings reach about \$30,100, a little better than the grid's best, with trees of 69 leaves, a learning rate of 0.07 and larger leaves (at least 44 rows each). The difference with the grid, about \$360, is half the spread across folds: the two searches have found equally good regions of settings, not one better model.

A first search is rarely the last. The usual practice is to look at the results, fix the hyperparameters whose best values agree, and search again in a narrower range around the best values of the rest; when the best value of a setting is at the edge of the range searched, the range goes further in that direction.

## Searching a pipeline

A search can tune any step of a [pipeline](../basic-workflows/california_with_pipelines.ipynb), with the name of the step, two underscores, and the name of the setting. Because the whole pipeline is refitted in every fold, the scaler is fitted on that fold's training rows only, and nothing leaks from the validation fold: `make_pipeline` names each step after its class, in lower case, and a setting of a step is written as the step's name, two underscores and the setting: `kneighborsregressor__n_neighbors`.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_search = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsRegressor()),
    param_grid={"kneighborsregressor__n_neighbors": [5, 10, 20, 40], "kneighborsregressor__weights": ["uniform", "distance"]},
    cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1,
).fit(X_rest, y_rest)
knn_search.best_params_, -knn_search.best_score_

({'kneighborsregressor__n_neighbors': 10,
  'kneighborsregressor__weights': 'distance'},
 np.float64(43752.24132233467))

The best k-NN uses 10 neighbours weighted by their distance, at about \$43,750: far behind gradient boosting on this table, but found with the same machinery, scaler included.

## The final test, once

After the search, `best_estimator_` is the best combination already retrained on all the rows given to the search. It is measured once on the test set:

In [ ]:
from sklearn.metrics import mean_absolute_error

{
    "grid search": mean_absolute_error(y_test, grid_search.best_estimator_.predict(X_test)),
    "randomised search": mean_absolute_error(y_test, random_search.best_estimator_.predict(X_test)),
}

{'grid search': 29244.879668895162, 'randomised search': 29503.953001311715}

Both tuned models miss the test districts by about \$29,200 to \$29,500, the grid's slightly ahead, which is the opposite of their order in cross-validation: differences of a few hundred dollars are within the noise of both measurements. What can be said is that tuning brought the test error from about \$31,000 with the default settings ([Measuring a regression model's error](../evaluation/regression_metrics.ipynb), on the same split) to about \$29,200.

The best cross-validated score of a search is optimistic for the same reason as in [Validation and model selection](validation_and_model_selection.ipynb#choosing-on-the-test-set): it is the best of many noisy estimates. The test set corrects for that. When even the test set is too small to spare, *nested* cross-validation runs the whole search inside each fold of an outer cross-validation, and estimates the error of the procedure (search included) at a much higher cost.